# LDTk example 1: basics

This first example covers the basics of LDTk. We learn how to set up filters, how to use ``LDPSetCreator`` to create a set of limb darkening profiles as instances of ``LDPSet`` class, how to use these instances to estimate limb darkening coefficients from the profiles, and how to evaluate the limb darkening model likelihood directly from the profiles.

In [1]:
%pylab inline
from IPython.display import display, Latex
import numpy as np
import seaborn as sb
sb.set_style('white')

Populating the interactive namespace from numpy and matplotlib


## Initialisation

In [2]:
import ldtk
ldtk.client.Client(update_server_file_list=True)
from ldtk import LDPSetCreator, TabulatedFilter

In [3]:
amp,tran  = np.genfromtxt('kepler_response_hires1.txt', unpack=True)
filters= [TabulatedFilter('kepler' ,amp,  tm=tran, tmf=1.)]


First, we initialise a ``LDPSetCreator`` with the stellar parameter estimates and our filter set. This may take some time, since we also need to download the necessary model spectra to a local cache directory (can be several hundreds of MB).

In [4]:
teff = (5400,50)
logg = (4.5,0.1)
z = (0.0,0.05)

sc = LDPSetCreator(teff=teff,logg=logg , z=z, filters=filters)

Next, we create the limb darkening profiles with their uncertainties for each filter, all contained in an ``LDPSet`` object.

In [5]:
ps = sc.create_profiles(nsamples=2000)
ps.resample_linear_z()

## Limb darkening coefficient estimation

The ``LDPSet`` class offers methods to estimate the limb darkening model coefficients for each filter. These methods are called ``coeffs_xx`` where ``xx`` can be

  - ``ln``:  linear model    (1 coeff)
  - ``qd``:  quadratic model (2 coeffs)
  - ``nl``:  nonlinear model (4 coeffs)
  - ``ge``:  general model   (n coeffs)
  
For the quadratic model

In [6]:
qc,qe = ps.coeffs_qd()

In [7]:
for i,(c,e) in enumerate(zip(qc,qe)):
    display(Latex('u$_{i:d} = {c[0]:5.4f} \pm {e[0]:5.4f}\quad$'
                  'v$_{i:d} = {c[1]:5.4f} \pm {e[1]:5.4f}$'.format(i=i+1,c=c,e=e)))

<IPython.core.display.Latex object>

We can also use MCMC to estimate the coefficient uncertainties (or the full covariance matrix) more accurately:

In [8]:
qc,qe = ps.coeffs_qd(do_mc=True, n_mc_samples=10000)

In [9]:
for i,(c,e) in enumerate(zip(qc,qe)):
    display(Latex('u$_{i:d} = {c[0]:5.4f} \pm {e[0]:5.4f}\quad$'
                  'v$_{i:d} = {c[1]:5.4f} \pm {e[1]:5.4f}$'.format(i=i+1,c=c,e=e)))

<IPython.core.display.Latex object>

Finally, we can decide we don't trust the stellar models the profiles were computed from too much, and set an multiplicative factor on the profile uncertainties. 

In [14]:
ps.set_uncertainty_multiplier(2)

qc,qe = ps.coeffs_qd(do_mc=True, n_mc_samples=10000)

for i,(c,e) in enumerate(zip(qc,qe)):
    display(Latex('u$_{i:d} = {c[0]:5.4f} \pm {e[0]:5.4f}\quad$'
                  'v$_{i:d} = {c[1]:5.4f} \pm {e[1]:5.4f}$'.format(i=i+1,c=c,e=e)))
    
print(qc)

<IPython.core.display.Latex object>

[[ 0.58497087  0.102373  ]]


### Likelihood evaluation

From down on I didnt change and it doesnt work
We can also directly evaluate the likelihood for a model. This can be done for a single passband

In [17]:
print(ps.lnlike_qd([0.69, 0.15], flt=0))

-3605.11819437


jointly for all passbands

In [18]:
print(ps.lnlike_qd([ 0.69, 0.15,   0.48 ,0.16,   0.38, 0.15]))
print(ps.lnlike_qd([[0.69, 0.15], [0.48 ,0.16], [0.38, 0.15]]))

AssertionError: 

or separately for all passbands

In [16]:
print(ps.lnlike_qd([ 0.69, 0.15,   0.48, 0.16,   0.38, 0.15],  joint=False))
print(ps.lnlike_qd([[0.69, 0.15], [0.48, 0.16], [0.38, 0.15]], joint=False))

[ 314.34169585  259.40711255  411.4290329 ]
[ 314.34169585  259.40711255  411.4290329 ]


---
<center> &copy; 2015 <a href="mailto:hannu.parviainen@physics.ox.ac.uk">Hannu Parviainen</a></center>